# 27. Calibration and performance diagnosis
A classifier can rank cases well but assign misleading probabilities. Calibration checks whether predicted probabilities match observed frequencies for comparable predictions. Learning curves compare performance at different training sizes. Validation curves compare model settings. Diagnose a problem before prescribing more data or a more complex model.

Use development partitions for calibration and diagnostics. CalibratedClassifierCV fits calibration without allowing calibration examples to directly train the corresponding base predictor. Inspect reliability curves and probability losses. Examine subgroup error counts and sample sizes; a small subgroup score is noisy. Explain possible harms and measurement limitations without treating group differences as causal findings.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X, y = load_breast_cancer(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
Xt, Xv, yt, yv = train_test_split(X_dev, y_dev, stratify=y_dev, random_state=7) # Reserve development validation examples.
calibrated = CalibratedClassifierCV(model, method="sigmoid", cv=3) # Calibrate through internal training folds.
calibrated.fit(Xt, yt) # Fit without final-test information.
p = calibrated.predict_proba(Xv)[:, 1] # Probabilities for class 1.
print("Validation Brier loss:", round(brier_score_loss(yv, p), 4)) # Evaluate squared probability error.
print("Probability range:", round(p.min(), 3), round(p.max(), 3)) # Inspect valid range.
assert ((p >= 0) & (p <= 1)).all() # Probabilities must be bounded.

## Learning curves and validation curves you can read
A learning curve changes the training-set size while keeping the learning procedure fixed. A validation curve changes one model setting. Read training and validation together: very strong training with weaker validation suggests sensitivity or overfitting; both weak suggests mismatch or insufficiently useful inputs, but neither pattern proves a unique diagnosis. More data is promising when validation improves with size, not merely because training accuracy is high. Curves are development diagnostics and must not consume a reserved final test.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, learning_curve, validation_curve
X, y = load_wine(return_X_y=True)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)) # Refit preprocessing for every curve point.
cv = StratifiedKFold(3, shuffle=True, random_state=42)
sizes, train, val = learning_curve(pipe, X, y, cv=cv, train_sizes=[0.4, 0.7, 1.0], scoring="accuracy", shuffle=True, random_state=42) # Vary training size.
values = [0.01, 0.1, 1., 10.]
vtrain, vval = validation_curve(pipe, X, y, param_name="logisticregression__C", param_range=values, cv=cv, scoring="accuracy") # Vary regularization setting.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(sizes, train.mean(axis=1), label="Training") # Average one result per fold at each size.
axes[0].plot(sizes, val.mean(axis=1), label="Validation")
axes[0].set(xlabel="Training examples", ylabel="Accuracy", title="Learning curve")
axes[1].semilogx(values, vtrain.mean(axis=1), label="Training")
axes[1].semilogx(values, vval.mean(axis=1), label="Validation")
axes[1].set(xlabel="C (inverse regularization strength)", ylabel="Accuracy", title="Validation curve")
for ax in axes: ax.legend()
fig.tight_layout(); fig.savefig("curves.png"); plt.close(fig) # Save both diagnostic plots.
print("Sizes:", sizes.tolist())
print("Learning validation means:", val.mean(axis=1).round(3).tolist())
print("Setting validation means:", vval.mean(axis=1).round(3).tolist())
assert train.shape == val.shape == (3, 3) # Three sizes and three folds.

### Reading the result
Each printed validation mean averages folds at a size or C setting. The graphs expose gaps hidden by a single average. These small Wine curves can have sampling noise and nonmonotonic points. Do not call a curve an unbiased final estimate after using it to select settings.

### Transfer exercise
Repeat with a shallow and a deep tree. Explain one hypothesis suggested by each curve and one alternative explanation.

## Your turn
Plot a calibration curve and a learning curve. Write a hypothesis about the main limitation, then name evidence that could disprove it.

Try the task before opening the solution on the course website.